# **2. Deep Q-Learning para _cartpole_**

En este cuaderno, vamos a resolver [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), un problema de control sencillo en el que el agente debe mantener un péndulo en equilibrio sobre un carro, moviéndolo a izquierda o derecha. A diferencia de Cliff Walking ([01_cliffwalking-qlearning.ipynb](01_cliffwalking-qlearning.ipynb)), el estado está formado por valores continuos, por lo que no podemos almacenar los valores $Q(s, a)$ en una tabla. En su lugar, usaremos **Deep Q-Learning** (DQN), que sustituye la tabla $Q$ por una red neuronal que aproxima $Q(s, a)$.

Basado en:
> Udacity, Deep Reinforcement Learning Course. Disponible en [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/dqn).

## **2.0. Preparación**

Comenzamos importando las bibliotecas necesarias, así como otras utilidades que emplearemos más adelante:

In [ ]:
import gymnasium as gym
import numpy as np
import torch
from collections import deque

import io
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image as IPImage
from PIL import Image

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def show_episode(env, act):
    """
    Ejecuta un episodio completo y lo muestra como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción a realizar en un estado
    """
    state, info = env.reset()
    frames = [Image.fromarray(env.render())]
    score = 0
    terminated = truncated = False

    # Bucle de interacción agente-entorno (1 episodio)
    while not (terminated or truncated):
        action = act(state)
        state, reward, terminated, truncated, info = env.step(action)
        frames.append(Image.fromarray(env.render()))
        score += reward

    # Guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format="gif", save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata["render_fps"], loop=0)
    display(IPImage(data=gif.getvalue(), format="gif"))
    print("Puntuación: ", score)

def plot_scores(scores, title, solved_score=None, window=100):
    """
    Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        scores (list): puntuación de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_score (float): puntuación media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    scores = np.asarray(scores)
    avg = [np.mean(scores[max(0, i-window+1):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label="Puntuación")
    plt.plot(np.arange(len(scores)), avg, label=f"Puntuación media (últimos {window})")
    if solved_score is not None:
        plt.axhline(solved_score, color="gray", linestyle="--", label="Resuelto")
    plt.ylabel("Puntuación")
    plt.xlabel("Episodio nº")
    plt.title(title)
    plt.legend()
    plt.show()

## **2.1. Entorno**

Creamos el entorno [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/). El estado es un vector de $4$ valores continuos (posición y velocidad del carro, ángulo y velocidad angular del péndulo) y hay $2$ acciones posibles: `0` (empujar el carro a la izquierda) y `1` (empujarlo a la derecha).

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

print(env.observation_space)
print(env.action_space)

Una vez creado el entorno, podemos visualizar el **estado inicial** de la simulación:

In [ ]:
# Inicializar el entorno
env.reset()

# Mostrar estado inicial del entorno
show_state(env)

Podemos interactuar con el entorno con un agente sin entrenar usando la clase `Agent` de [dqn_agent.py](dqn_agent.py). La función `show_episode()` muestra el episodio como una animación, junto con su **puntuación**, es decir, la suma de las recompensas obtenidas a lo largo del episodio:

In [ ]:
from dqn_agent import Agent

agent_untrained = Agent(state_size=4, action_size=2, seed=0)

show_episode(env, agent_untrained.act)

## **2.2. Algoritmo DQN**

A continuación, proporcionamos una implementación genérica de Deep Q-Learning (DQN). Nos apoyamos en ficheros externos:

*   [dqn_agent.py](dqn_agent.py): Implementación de las capacidades del agente (`act`, `step`, etc.)
*   [model.py](model.py): Implementación de la red neuronal del agente (la red Q)

El entorno [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) se considera resuelto cuando el agente obtiene una puntuación media de al menos $475$ puntos en 100 episodios consecutivos. Los episodios se truncan a los $500$ pasos, que es por tanto la puntuación máxima.

In [ ]:
SOLVED_SCORE = 475.0   # Puntuación media (últimos 100 episodios) para considerar resuelto CartPole-v1

def dqn(agent, n_episodes=2000, eps_start=1.0, eps_end=0.01, eps_decay=0.995, print_every=100):
    """Deep Q-Learning.

    Parámetros
    ==========
        agent (Agent): agente a entrenar
        n_episodes (int): número máximo de episodios
        eps_start (float): valor inicial de epsilon
        eps_end (float): valor final de epsilon
        eps_decay (float): factor de multiplicación de epsilon (por episodio)
        print_every (int): mostrar la puntuación cada print_every episodios
    """
    scores = []                        # Puntuaciones de todos los episodios
    scores_window = deque(maxlen=100)  # Puntuaciones de los últimos 100 episodios
    eps = eps_start                    # Inicializar epsilon

    for i_episode in range(1, n_episodes+1):
        state, info = env.reset()
        score = 0
        terminated = truncated = False

        # Bucle de interacción agente-entorno (1 episodio)
        while not (terminated or truncated):
            # Elegir la acción At con la política ε-greedy
            action = agent.act(state, eps)

            # Aplicar At y obtener Rt+1, St+1
            next_state, reward, terminated, truncated, info = env.step(action)

            # Almacenar <St, At, Rt+1, St+1>, entrenar y actualizar
            # (solo 'terminated' marca St+1 como terminal; tras un truncamiento hay que seguir haciendo bootstrapping)
            agent.step(state, action, reward, next_state, terminated)

            # Avanzar al siguiente estado
            state = next_state
            score += reward

        scores_window.append(score)       # Guardar la última puntuación (ventana de 100)
        scores.append(score)              # Guardar la última puntuación (todos los episodios)
        eps = max(eps_end, eps_decay*eps) # Reducir epsilon

        # Mostrar información del entrenamiento
        if i_episode % print_every == 0:
            print(f"Episodio {i_episode}\tPuntuación media (últimos 100): {np.mean(scores_window):.2f}")
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print(f"\n¡Problema resuelto en {i_episode-100} episodios!\tPuntuación media (últimos 100): {np.mean(scores_window):.2f}")
            break
    else:
        print(f"\nProblema no resuelto en {n_episodes} episodios.\tPuntuación media (últimos 100): {np.mean(scores_window):.2f}")

    torch.save(agent.qnetwork_local.state_dict(), "checkpoint-cartpole-dqn.pth") # Guardar los pesos del agente entrenado

    return scores

agent = Agent(state_size=4, action_size=2, seed=0)
scores = dqn(agent)

plot_scores(scores, "DQN", SOLVED_SCORE)

## **2.3. Visualizar el agente entrenado**

Cargamos los pesos en el agente y lo ejecutamos durante un episodio con la política _greedy_:

In [ ]:
# Cargar los pesos desde `checkpoint-cartpole-dqn.pth`
agent_load = Agent(state_size=4, action_size=2, seed=0)

agent_load.qnetwork_local.load_state_dict(torch.load("checkpoint-cartpole-dqn.pth", weights_only=True))

show_episode(env, agent_load.act)

¡Nuestro agente ya sabe mantener el péndulo en equilibrio!

En este cuaderno, hemos sustituido la tabla $Q$ por una red neuronal (DQN) para resolver un problema con estados continuos, en el que una tabla no es viable.

En el próximo cuaderno, reutilizaremos el mismo agente DQN, sin ningún cambio, para resolver de nuevo Cliff Walking.